# EEGConformer — BCI-IV-2a (Neuro-Fabric)

Trains and exports the production `eegconformer.onnx` artefact.

**Runtime:** GPU (T4 is enough). **Wall clock:** ~30–60 min.


## 1. Clone Neuro-Fabric and install deps


In [ ]:
import os, subprocess, sys
REPO = os.environ.get('NEUROFABRIC_REPO', 'https://github.com/your-org/neuro-fabric.git')
BRANCH = os.environ.get('NEUROFABRIC_BRANCH', 'main')
if not os.path.isdir('/content/repo'):
    subprocess.check_call(['git','clone','--depth','1','--branch',BRANCH,REPO,'/content/repo'])
%cd /content/repo/training
!python -m pip install -q -r requirements.txt


## 2. Verify GPU


In [ ]:
import torch; print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')


## 3. Review the contract (`configs/eegconformer-bciiv2a.yaml`)


In [ ]:
!cat configs/eegconformer-bciiv2a.yaml


## 4. Acquire BCI-IV-2a


In [ ]:
!python scripts/acquire_dataset.py


## 5. Preprocess to (22, 1000) @ 250 Hz


In [ ]:
!python scripts/preprocess.py


## 6. Train EEGConformer


In [ ]:
!python scripts/train.py


## 7. Validate (cross-subject hold-out)


In [ ]:
!python scripts/validate.py


## 8. Evaluate embeddings (cosine recall@10)


In [ ]:
!python scripts/evaluate.py


## 9. Export to ONNX (opset 17, parity ≥ 0.999)


In [ ]:
!python scripts/export_onnx.py


## 10. Package artefact (manifest + MODEL_CARD)


In [ ]:
!python scripts/package.py
!ls -lah artefacts/eegconformer-bciiv2a-v1


## 11. Download the artefact


In [ ]:
import shutil, glob, os
root = 'artefacts/eegconformer-bciiv2a-v1'
out = shutil.make_archive('/content/eegconformer-bciiv2a-v1', 'zip', root)
print('zip:', out, os.path.getsize(out), 'bytes')
from google.colab import files
files.download(out)


## 12. Deploy into Neuro-Fabric

Upload `eegconformer.onnx` to `public/models/` (or Lovable Cloud storage), then at app boot:

```ts
import { registerBraindecodeEEGConformer } from '@/lib/ai/models/registry';
registerBraindecodeEEGConformer({ artifact: { kind: 'url', url: '/models/eegconformer.onnx' } });
```
